In [ ]:
# ==========================================
# ## 5. 머신러닝 모델 학습 및 평가 (Modeling & Evaluation)
# ==========================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

# 한글 폰트 및 시각화 세팅
plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

print("=== 🔄 5-0. 저장된 전처리 완료 데이터셋 로드 ===")
df_encoded = pd.read_csv('./data/hnm_churn_processed.csv')
print(f"▶ 데이터셋 크기 확인: {df_encoded.shape}")

# 독립변수와 타겟 정의
X = df_encoded.drop(columns=['churn'])
y = df_encoded['churn']

# 정석 분할 (8:2)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("▶ Train/Validation 세트 분할 완료!")
print("-" * 60)


# ------------------------------------------
# 5-1. LightGBM 모델 정의 및 하이퍼파라미터 설정
# ------------------------------------------
lgb_params = {
    'learning_rate': 0.05,
    'max_depth': 10,
    'num_leaves': 63,
    'subsample': 0.7,
    'colsample_bytree': 0.8,
    'n_estimators': 1000,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

model_lgb = lgb.LGBMClassifier(**lgb_params)


# ------------------------------------------
# 5-2. 모델 학습 (Early Stopping 적용)
# ------------------------------------------
print("=== 🚀 5-2. LightGBM 모델 학습 (Early Stopping 반영) ===")
model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
)
print("✅ 조기 종료 적용 학습 완료!")


# ------------------------------------------
# 5-3. 검증 데이터셋 기반 성능 평가
# ------------------------------------------
print("\n=== 📊 5-3. 검증 데이터셋 기반 종합 성능 평가 ===")
y_pred = model_lgb.predict(X_val)
y_proba = model_lgb.predict_proba(X_val)[:, 1]

print("-" * 60)
print(f"🎯 검증 데이터셋 정확도 (Accuracy) : {accuracy_score(y_val, y_pred) * 100:.2f}%")
print(f"✨ 모델 순수 분별력 (AUC-ROC)  : {roc_auc_score(y_val, y_proba):.4f}")
print("-" * 60)

# 혼동 행렬 시각화
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['유지(0)', '이탈(1)'], yticklabels=['유지(0)', '이탈(1)'])
plt.title("🎯 혼동 행렬 (Confusion Matrix) 결과", fontsize=12, weight='bold', pad=10)
plt.xlabel("모델의 예측값 (Predicted)")
plt.ylabel("실제 정답 라벨 (Actual)")


# ------------------------------------------
# 5-4. 피처 중요도(Feature Importance) 시각화 및 해석
# ------------------------------------------
print("\n=== 👗 5-4. 피처 중요도(Feature Importance) 분석 ===")
df_fi = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model_lgb.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.subplot(1, 2, 2)
sns.barplot(data=df_fi.head(15), x='Importance', y='Feature', palette='viridis')
plt.title("🔥 이탈 예측 기여도 Top 15 피처 (Feature Importance)", fontsize=12, weight='bold', pad=10)
plt.xlabel("기여도 점수 (Gain/Split)")
plt.ylabel("파생 변수명")

plt.tight_layout()
plt.show()

## 5. 머신러닝 모델 학습 및 평가 (Modeling & Evaluation)
5-1. LightGBM 모델 정의 및 하이퍼파라미터 설정
5-2. 모델 학습 (Early Stopping 적용)
5-3. 검증 데이터셋 기반 성능 평가 (Accuracy, AUC-ROC, 혼동 행렬 확인)
5-4. 피처 중요도(Feature Importance) 시각화 및 해석